In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

## 1. Imports and Model Loading

In [ ]:
import os
import imageio
import uuid
from IPython.display import Image as ImageDisplay

import os, sys
import numpy as np
# Point to CUDA 12.8
os.environ["CUDA_HOME"] = "/usr/local/cuda-12.8"
os.environ["PATH"] = f"/usr/local/cuda-12.8/bin:{os.environ['PATH']}"
os.environ["FORCE_CUDA"] = "1"
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
print("CUDA_HOME", os.environ.get("CUDA_HOME"))
print("TORCH_EXTENSIONS_DIR", os.environ.get("TORCH_EXTENSIONS_DIR"))
from torch.utils import cpp_extension
print("include_paths", cpp_extension.include_paths("cuda"))
# Use the same extensions dir you built to (adjust if you prefer a single location)
os.environ["TORCH_EXTENSIONS_DIR"] = "/home/mnc/mccv/sam-3d-objects/.torch_extensions"
os.makedirs(os.environ["TORCH_EXTENSIONS_DIR"], exist_ok=True)

# Notebook path setup
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.insert(0, os.getcwd())
from gsplat.cuda import _backend
print("gsplat CUDA loaded:", hasattr(_backend, "_C"))



from inference import Inference, ready_gaussian_for_video_rendering, render_video, load_image, load_masks, load_single_mask, display_image, make_scene, interactive_visualizer, debug_inference_and_save

In [ ]:
PATH = os.getcwd()
TAG = "hf"
config_path = f"{PATH}/../checkpoints/{TAG}/pipeline.yaml"
inference = Inference(config_path, compile=False)

## 2. Load input image to lift to 3D (single object)

In [ ]:
IMAGE_PATH = f"{PATH}/images/shutterstock_stylish_kidsroom_1640806567/image.png"
IMAGE_NAME = os.path.basename(os.path.dirname(IMAGE_PATH))

image = load_image(IMAGE_PATH)
mask = load_single_mask(os.path.dirname(IMAGE_PATH), index=14)
display_image(image, masks=[mask])

In [ ]:
# Use the recently generated mask from demo_single_object_sam2_click.py
IMAGE_PATH = "/home/mnc/Downloads/IMG_3792.jpg"
IMAGE_NAME = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
MASK_DIR = "/home/mnc/Downloads"

image = load_image(IMAGE_PATH)
mask = load_single_mask(MASK_DIR, index=0)
display_image(image, masks=[mask])


In [ ]:
# Bird image: load and prepare mask (auto-SAM if available)
BIRD_IMAGE_PATH = "/home/mnc/Downloads/Robin.png"
BIRD_IMAGE_NAME = os.path.splitext(os.path.basename(BIRD_IMAGE_PATH))[0]

bird_image = load_image(BIRD_IMAGE_PATH)

# Try to auto-segment the bird with SAM. If unavailable, fall back to full image mask.
bird_mask = None
try:
    from segment_anything import SamAutomaticMaskGenerator, sam_model_registry
    # TODO: point SAM_CHECKPOINT to your downloaded SAM weights (e.g., sam_vit_h_4b8939.pth).
    SAM_CHECKPOINT = os.environ.get("SAM_CHECKPOINT", "/path/to/sam_checkpoint.pth")
    sam_model = sam_model_registry["vit_h"](checkpoint=SAM_CHECKPOINT)
    sam_model.to("cuda" if torch.cuda.is_available() else "cpu")
    mask_generator = SamAutomaticMaskGenerator(sam_model)
    masks = mask_generator.generate(bird_image)
    if len(masks) > 0:
        # pick the largest mask (most likely the bird)
        bird_mask = max(masks, key=lambda m: m.get("area", 0)).get("segmentation")
        print("SAM mask selected with area", masks[0].get("area", 0))
    else:
        print("SAM produced no masks; using full-image mask.")
except Exception as e:
    print(f"SAM auto-mask unavailable: {e}. Using full-image mask.")

if bird_mask is None:
    bird_mask = np.ones(bird_image.shape[:2], dtype=bool)

display_image(bird_image, masks=[bird_mask])


In [ ]:
# Run inference on bird image
bird_output = debug_inference_and_save(
    inference_fn=lambda img, msk, seed=42: inference(img, msk, seed=seed),
    image=bird_image,
    mask=bird_mask,
    out_dir=f"{PATH}/gaussians/single",
    image_name=BIRD_IMAGE_NAME,
    seed=42,
)

# Reuse visualization cells below with the bird output
IMAGE_NAME = BIRD_IMAGE_NAME
output = bird_output


In [ ]:
# Render bird Gaussian splat to GIF
os.makedirs(f"{PATH}/gaussians/single", exist_ok=True)

bird_scene = make_scene(bird_output)
bird_scene = ready_gaussian_for_video_rendering(bird_scene)
bird_video = render_video(bird_scene, r=1, fov=60, pitch_deg=15, yaw_start_deg=-45, resolution=512)["color"]

bird_gif_path = os.path.join(PATH, "gaussians", "single", f"{BIRD_IMAGE_NAME}.gif")
imageio.mimsave(bird_gif_path, bird_video, format="GIF", duration=1000/30, loop=0)

ImageDisplay(url=f"gaussians/single/{BIRD_IMAGE_NAME}.gif?cache_invalidator={uuid.uuid4()}") 




In [ ]:
# Interactive view of bird Gaussian splat (may take a moment to load)
interactive_visualizer(f"{PATH}/gaussians/single/{BIRD_IMAGE_NAME}.ply")


## 3. Generate Gaussian Splat

In [ ]:
# run model
output = debug_inference_and_save(
    inference_fn=lambda img, msk, seed=42: inference(img, msk, seed=seed),
    image=image,
    mask=mask,
    out_dir=f"{PATH}/gaussians/single",
    image_name=IMAGE_NAME,
    seed=42,
)

# export gaussian splat (as point cloud)
# (saved above by debug_inference_and_save)


## 4. Visualize Gaussian Splat
### a. Animated Gif

In [ ]:
# render gaussian splat
scene_gs = make_scene(output)
scene_gs = ready_gaussian_for_video_rendering(scene_gs)

video = render_video(
    scene_gs,
    r=1,
    fov=60,
    pitch_deg=15,
    yaw_start_deg=-45,
    resolution=512,
)["color"]

# save video as gif
imageio.mimsave(
    os.path.join(f"{PATH}/gaussians/single/{IMAGE_NAME}.gif"),
    video,
    format="GIF",
    duration=1000 / 30,  # default assuming 30fps from the input MP4
    loop=0,  # 0 means loop indefinitely
)

# notebook display
ImageDisplay(url=f"gaussians/single/{IMAGE_NAME}.gif?cache_invalidator={uuid.uuid4()}")

In [ ]:
import os, uuid, imageio

# ensure folder exists
os.makedirs(f"{PATH}/gaussians/single", exist_ok=True)

scene_gs = make_scene(output)
scene_gs = ready_gaussian_for_video_rendering(scene_gs)
video = render_video(scene_gs, r=1, fov=60, pitch_deg=15, yaw_start_deg=-45, resolution=512)["color"]

gif_path = os.path.join(PATH, "gaussians", "single", f"{IMAGE_NAME}.gif")
imageio.mimsave(gif_path, video, format="GIF", duration=1000/30, loop=0)

ImageDisplay(url=f"gaussians/single/{IMAGE_NAME}.gif?cache_invalidator={uuid.uuid4()}")


### b. Interactive Visualizer

In [ ]:
# might take a while to load (black screen)
interactive_visualizer(f"{PATH}/gaussians/single/{IMAGE_NAME}.ply")